# 00 · Setup de datos — SEPA

Prepara el entorno y la base para el resto de los notebooks. Dos modos:

- **Modo BUILD (Santiago):** monta Google Drive / usa la copia local con `base_sepa.zip`,
  valida el esquema y construye el Parquet particionado (o un mes de muestra).
- **Modo LECTOR:** descarga los *agregados livianos* ya procesados (cuando estén publicados)
  para reproducir los análisis sin bajar los 8,5 GB.

> Repo: https://github.com/santiagoriverti/precios_supermercados_argentina · ver `docs/ARQUITECTURA.md`.


## 1. Entorno

In [ ]:
# Instalar dependencias (en Colab)
try:
    import google.colab  # noqa
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    !pip -q install duckdb pyarrow gdown pyyaml
print("Colab:", EN_COLAB)

In [ ]:
# Traer el codigo del repo (paquete precios_sepa + config) cuando se corre en Colab
import sys, os
from pathlib import Path

REPO_URL = "https://github.com/santiagoriverti/precios_supermercados_argentina.git"
if EN_COLAB:
    if Path("precios_supermercados_argentina").exists():
        !cd precios_supermercados_argentina && git pull -q   # traer siempre la ultima version
    else:
        !git clone -q $REPO_URL
    REPO = Path("precios_supermercados_argentina").resolve()
else:
    # Local: subir hasta la raiz del repo desde notebooks/
    REPO = Path.cwd()
    while REPO.name and not (REPO / "src" / "precios_sepa").exists():
        REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)
import precios_sepa
print("Repo:", REPO, "| precios_sepa", precios_sepa.__version__)

## 2. Ubicar la base

`base_sepa.zip` (~8,5 GB) vive en Google Drive. Opciones:
- **A) Copia local ya extraida** (PC de Santiago): apuntar a la carpeta.
- **B) Montar Drive** en Colab y leer el zip desde "Mi unidad".
- **C) gdown** con el ID publico (completar en `config/settings.yml`).

In [ ]:
cfg = precios_sepa.load_settings()

# --- Elegir fuente de datos ---
BASE_DIR = None

# A) Copia local ya extraida (ajustar si hace falta)
_local = Path(cfg["rutas"]["base_extraida"])
if _local.exists():
    BASE_DIR = _local

# B) Montar Google Drive (Colab)
if BASE_DIR is None and EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Ajustar la ruta a donde este base_sepa.zip / la carpeta extraida en tu Drive
    _drive_zip = Path("/content/drive/MyDrive/base_sepa.zip")
    _drive_dir = Path("/content/drive/MyDrive/base_sepa")
    if _drive_dir.exists():
        BASE_DIR = _drive_dir
    elif Path("/content/base_sepa").exists():
        BASE_DIR = Path("/content/base_sepa")            # ya extraido (evita re-extraer al reiniciar)
    elif _drive_zip.exists():
        import zipfile
        print("Extrayendo base_sepa.zip (una vez, tarda unos minutos)...")
        with zipfile.ZipFile(_drive_zip) as z:
            z.extractall("/content/base_sepa")
        BASE_DIR = Path("/content/base_sepa")

assert BASE_DIR and BASE_DIR.exists(), "No se encontro la base. Configura BASE_DIR."
print("BASE_DIR =", BASE_DIR)

## 3. Inventario de archivos
Descubre los archivos, clasifica por tipo/mes y resuelve el duplicado `2024 bis`.

In [ ]:
import pandas as pd
from precios_sepa.io import descubrir_archivos

archivos = descubrir_archivos(BASE_DIR)
inv = pd.DataFrame([{
    "tipo": a.tipo, "periodo": a.periodo, "parte": a.parte,
    "bis": a.es_bis, "MB": round(a.path.stat().st_size/1e6, 1), "archivo": a.path.name,
} for a in archivos])

print(f"{len(archivos)} archivos ({inv.tipo.value_counts().to_dict()})")
print("Periodos:", inv.periodo.min(), "->", inv.periodo.max())
resumen = inv.groupby(["tipo","periodo"]).parte.nunique().unstack("tipo").fillna(0).astype(int)
resumen.tail(12)

## 4. Validar el esquema
Confirma columnas, unidad de precios (pesos vs. centavos) y factor autodetectado.

In [ ]:
import gzip, numpy as np
from precios_sepa.clean import limpiar_precio, detectar_factor_precio

def _peek(archivo, nrows=200_000):
    with gzip.open(archivo.path, "rt") as g:
        return pd.read_csv(g, dtype={"id_producto":"str","id_comercio":"str","id_bandera":"str"}, nrows=nrows)

a_min = next(a for a in archivos if a.tipo == "minorista")
a_may = next(a for a in archivos if a.tipo == "mayorista")

for nombre, a in [("MINORISTA", a_min), ("MAYORISTA", a_may)]:
    df = _peek(a)
    pcols = [c for c in df.columns if c.startswith("precio_")]
    print(f"== {nombre} {a.periodo} ==  cols_id={list(df.columns[:5])}")
    print(f"   {len(pcols)} columnas de precio, ej: {pcols[:4]}")
    # factor sobre la 1er columna de precio
    tmp = df.rename(columns={pcols[0]: "precio"})
    tmp["precio"] = limpiar_precio(tmp["precio"])
    try:
        f = detectar_factor_precio(tmp)
        print(f"   factor detectado = {f}  (1=pesos, 100=centavos) | mediana={tmp['precio'].median():.1f}")
    except Exception as e:
        print("   factor:", e)
    print()

## 5. (Opcional) Construir un mes de muestra

Convierte **un** mes a Parquet particionado para probar el pipeline sin correr las horas del
build completo. El build completo se corre en local con:

```bash
python scripts/00_build_parquet.py --base "<carpeta base_sepa>"
```

In [ ]:
from precios_sepa.ingest import procesar_archivo

# Demo LIVIANO para Colab: muestra CAPEADA de un archivo minorista y uno mayorista.
# En Colab gratuito (~12 GB RAM) NO conviene construir meses enteros: el build COMPLETO
# se corre en tu PC ->  python scripts/00_build_parquet.py --base "<carpeta base_sepa>"
OUT = REPO / "data" / "interim" / "sepa"
procesar_archivo(a_min, OUT, chunksize=100_000, limite_filas=300_000)   # minorista: 15 cols/dia
procesar_archivo(a_may, OUT, chunksize=50_000,  limite_filas=200_000)   # mayorista: 60 cols/dia (mas pesado)
print("Muestra construida en", OUT)

In [ ]:
# Verificar con DuckDB
import duckdb
glob = str(OUT / "**" / "*.parquet")
con = duckdb.connect()
q = con.sql(f"""
    SELECT tipo, count(*) AS filas, count(DISTINCT id_producto) AS productos,
           count(DISTINCT id_sucursal) AS sucursales, min(fecha) AS desde, max(fecha) AS hasta
    FROM read_parquet('{glob}', hive_partitioning=1, union_by_name=1)
    GROUP BY tipo ORDER BY tipo
""").df()
q

## Listo

- Esquema validado y (opcional) un mes en Parquet.
- Seguí con **`01_exploracion_base.ipynb`** para la cobertura, calidad y el mapa de sucursales.

**Pendiente para el modo LECTOR:** publicar los agregados livianos y completar
`drive.agregados_publicos_id` en `config/settings.yml`.